# Technical Feature Pruning: $1T Smoke EDA

This first gated notebook performs target-agnostic diagnostics only. It builds the production pandas-ta-classic families from local Quant Warehouse prices, measures runtime and memory, rejects broken/constant columns, and identifies highly redundant columns. It does not use a linear model or predictive performance to discard indicators.

In [1]:
from __future__ import annotations

from pathlib import Path
from time import perf_counter
import os
import sys

import numpy as np
import polars as pl
from IPython.display import display

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from quant_warehouse.platforms.data_providers.fmp.feature_engineering import build_price_ta_classic_feature_families
from quant_warehouse.research_tools import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse
MIN_MARKET_CAP = 1_000_000_000_000
START_DATE = '2018-01-01'
MAX_CORRELATION_ROWS = 30_000
CORRELATION_THRESHOLD = 0.995
MIN_COVERAGE = 0.80

def rss_gib() -> float:
    fields = Path('/proc/self/status').read_text().splitlines()
    value = next(line for line in fields if line.startswith('VmRSS:')).split()[1]
    return float(value) / 1024 / 1024

print({'python': sys.version.split()[0], 'architecture': os.uname().machine, 'rss_gib': round(rss_gib(), 3)})

{'python': '3.13.11', 'architecture': 'aarch64', 'rss_gib': 0.301}


## Universe and incremental family construction

In [2]:
warehouse = Warehouse()
config = FamilyEvaluationConfig(market_cap_min=MIN_MARKET_CAP, start_date=START_DATE, max_features_per_family=None)
symbols, raw_universe, eligibility, universe_source = screen_fmp_equity_universe(config, warehouse=warehouse, required_sections=('prices',))
print({'universe_source': universe_source, 'symbols': len(symbols), 'eligible': int(eligibility['eligible'].sum())})
display(eligibility['reason'].value_counts().rename_axis('reason').reset_index(name='symbols').head(15))

family_parts: dict[str, list[pl.DataFrame]] = {}
timing_rows = []
peak_rss = rss_gib()
started = perf_counter()
for number, symbol in enumerate(symbols, start=1):
    prices = warehouse.read_prices(symbol, provider='fmp', start=START_DATE)
    if prices is None or prices.empty:
        timing_rows.append({'symbol': symbol, 'status': 'missing_prices'})
        continue
    symbol_started = perf_counter()
    built = build_price_ta_classic_feature_families(symbol, prices)
    for family, result in built.items():
        if result.df.empty or not result.feature_cols:
            continue
        frame = result.df[result.feature_cols].reset_index()
        family_parts.setdefault(family, []).append(frame)
    current_rss = rss_gib()
    peak_rss = max(peak_rss, current_rss)
    timing_rows.append({'symbol': symbol, 'status': 'ok', 'price_rows': len(prices), 'seconds': perf_counter() - symbol_started, 'rss_gib': current_rss})
    if number == 1 or number % 20 == 0 or number == len(symbols):
        print(f'{number}/{len(symbols)} {symbol} elapsed={perf_counter()-started:.1f}s rss={current_rss:.2f} GiB')

timings = pl.DataFrame(timing_rows)
print({'build_seconds': round(perf_counter()-started, 2), 'peak_rss_gib': round(peak_rss, 3), 'families': len(family_parts)})
display(timings.describe(include='all'))

{'universe_source': 'openbb:fmp', 'symbols': 14, 'eligible': 14}


,reason,symbols
0,ok,14


1/14 AAPL elapsed=1.1s rss=0.63 GiB


14/14 TSLA elapsed=13.0s rss=0.67 GiB
{'build_seconds': 12.98, 'peak_rss_gib': 0.673, 'families': 6}


,symbol,status,price_rows,seconds,rss_gib
count,14,14,14.000000,14.000000,14.000000
unique,14,1,NaN,NaN,NaN
top,AAPL,ok,NaN,NaN,NaN
freq,1,14,NaN,NaN,NaN
mean,NaN,NaN,1989.642857,0.924649,0.656151
std,NaN,NaN,566.326572,0.226870,0.014491
min,NaN,NaN,22.000000,0.147168,0.628391
25%,NaN,NaN,2141.000000,0.970261,0.645739
50%,NaN,NaN,2141.000000,0.972005,0.657280
75%,NaN,NaN,2141.000000,0.976240,0.668602


## Safe quality gates and redundancy clusters

In [3]:
quality_rows = []
redundancy_rows = []
for family, parts in sorted(family_parts.items()):
    panel = pl.concat(parts, ignore_index=True, sort=False)
    features = [column for column in panel.columns if column not in {'date', 'symbol'}]
    numeric = panel[features].apply(pl.Series.cast, errors='coerce').replace([np.inf, -np.inf], np.nan)
    for feature in features:
        values = numeric[feature]
        finite = values.dropna()
        coverage = float(values.notna().mean())
        unique = int(finite.nunique())
        variance = float(finite.var()) if len(finite) > 1 else np.nan
        if coverage < MIN_COVERAGE:
            status = 'drop_low_coverage'
        elif unique <= 1 or not np.isfinite(variance) or variance == 0:
            status = 'drop_constant_or_invalid'
        else:
            status = 'candidate'
        quality_rows.append({'family': family, 'feature': feature, 'coverage': coverage, 'unique_values': unique, 'variance': variance, 'status': status})
    candidates = [row['feature'] for row in quality_rows if row['family'] == family and row['status'] == 'candidate']
    sample = numeric[candidates]
    if len(sample) > MAX_CORRELATION_ROWS:
        sample = sample.sample(MAX_CORRELATION_ROWS, random_state=20260711)
    corr = sample.corr(method='spearman', min_periods=500).abs()
    coverage_lookup = numeric[candidates].notna().mean().to_dict()
    kept = []
    for feature in sorted(candidates, key=lambda value: (-coverage_lookup[value], value)):
        duplicate_of = next((other for other in kept if corr.at[feature, other] >= CORRELATION_THRESHOLD), None)
        if duplicate_of is None:
            kept.append(feature)
        else:
            redundancy_rows.append({'family': family, 'feature': feature, 'representative': duplicate_of, 'abs_spearman': float(corr.at[feature, duplicate_of])})

quality = pl.DataFrame(quality_rows)
redundancy = pl.DataFrame(redundancy_rows)
summary = quality.groupby(['family', 'status']).size().unstack(fill_value=0)
summary['redundant_candidates'] = redundancy.groupby('family').size().reindex(summary.index, fill_value=0) if not redundancy.empty else 0
summary['safe_survivors'] = summary.get('candidate', 0) - summary['redundant_candidates']
display(summary)
display(quality.loc[quality['status'].ne('candidate')].sort_values(['status', 'coverage']).head(50))
display(redundancy.sort_values(['family', 'abs_spearman'], ascending=[True, False]).head(80) if not redundancy.empty else redundancy)
print({'raw_features': len(quality), 'safe_candidates': int(quality['status'].eq('candidate').sum()), 'redundant': len(redundancy), 'safe_survivors': int(quality['status'].eq('candidate').sum()-len(redundancy)), 'final_rss_gib': round(rss_gib(), 3)})

status,candidate,drop_constant_or_invalid,redundant_candidates,safe_survivors
family,,,,
technical_candles,69,3,7,62
technical_cycles,11,0,0,11
technical_math,25,0,11,14
technical_momentum,170,0,30,140
technical_overlap,112,0,84,28
technical_performance,9,0,4,5


,family,feature,coverage,unique_values,variance,status
7,technical_candles,ta_candle__pattern_all_cdl_3_starsinsouth,1.0,1,0.0,drop_constant_or_invalid
14,technical_candles,ta_candle__pattern_all_cdl_concealbabyswall,1.0,1,0.0,drop_constant_or_invalid
33,technical_candles,ta_candle__pattern_all_cdl_identical_3_crows,1.0,1,0.0,drop_constant_or_invalid


,family,feature,representative,abs_spearman
3,technical_candles,ta_candle__pattern_all_cdl_doji_10_0_1,ta_candle__doji_cdl_doji_10_0_1,1.000000
4,technical_candles,ta_candle__pattern_all_cdl_inside,ta_candle__inside_cdl_inside,1.000000
5,technical_candles,ta_candle__pattern_all_cdl_kickingbylength,ta_candle__pattern_all_cdl_kicking,1.000000
0,technical_candles,ta_candle__ha_high,ta_candle__ha_close,0.999884
1,technical_candles,ta_candle__ha_low,ta_candle__ha_close,0.999862
...,...,...,...,...
79,technical_overlap,ta_overlap__mcgd_10,ta_overlap__avgprice,0.998155
107,technical_overlap,ta_overlap__vidya_14,ta_overlap__aberration_aber_sg_5_15,0.998058
123,technical_overlap,ta_overlap__mmar_10,ta_overlap__aberration_aber_sg_5_15,0.997851
64,technical_overlap,ta_overlap__hilo_hil_os_13_21,ta_overlap__alma_20_6_0_0_85,0.997829


{'raw_features': 399, 'safe_candidates': 396, 'redundant': 136, 'safe_survivors': 260, 'final_rss_gib': 0.678}


## Gate

Only broken, constant, or near-duplicate outputs are candidates for mechanical removal. Predictive selection is deferred to nonlinear anchored annual WFO in Quant Orchestrator. The measured runtime and memory from this notebook determine whether the next $1T target-quality notebook is safe to run.